### 1. Research Environment Setup

BasaLens relies on a combination of machine learning, multilingual data processing, and evaluation tools. This step installs the dependencies required to conduct the experiment and develop the interactive translation interface.

Key components include Hugging Face Transformers for NLLB-200 inference, Datasets for accessing FLORES+, SacreBLEU for chrF evaluation, and Gradio for presenting the model through an interactive demonstration. The Transformers version is constrained to below version 5 to maintain compatibility with the project's implementation.

In [ ]:
!pip -q install -U "transformers<5" datasets sacrebleu accelerate sentencepiece gradio

### 2. Experimental Environment and Reproducibility

This stage establishes the computational environment for the BasaLens evaluation. The experiment uses a fixed random seed (`42`) to maintain consistency in data sampling and model execution, supporting reproducibility across runs.

The required libraries provide the foundation for dataset handling, NLLB-200 inference, and chrF-based evaluation. GPU availability is also checked to determine whether accelerated inference can be used. Hugging Face authentication enables access to the pretrained model and research dataset.

In [ ]:
import random
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sacrebleu.metrics import CHRF
from IPython.display import display

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

login()

### 3. Constructing the Parallel Evaluation Dataset

To ensure a consistent comparison between English-to-Indonesian and English-to-Balinese translation, BasaLens uses the `devtest` split of the FLORES+ multilingual benchmark dataset.

The English, Indonesian, and Balinese language configurations are loaded separately and aligned using their shared sentence IDs. A one-to-one merge validation helps prevent mismatched or duplicated sentence pairs from entering the evaluation.

From the aligned dataset, 100 sentence pairs are selected using the fixed random seed (`42`). This creates a consistent evaluation sample for both translation directions, allowing their results to be compared under the same input conditions.

In [ ]:
DATASET_NAME = 'openlanguagedata/flores_plus'

def load_language_split(config_name: str, output_column: str) -> pd.DataFrame:
    dataset = load_dataset(DATASET_NAME, config_name, split='devtest')
    frame = dataset.select_columns(['id', 'text']).to_pandas()
    return frame.rename(columns={'text': output_column})

english_df = load_language_split('eng_Latn', 'english')
indonesian_df = load_language_split('ind_Latn', 'indonesian_reference')
balinese_df = load_language_split('ban_Latn', 'balinese_reference')

aligned = english_df.merge(indonesian_df, on='id', how='inner', validate='one_to_one')
aligned = aligned.merge(balinese_df, on='id', how='inner', validate='one_to_one')

if len(aligned) < 100:
    raise ValueError(f'Expected at least 100 aligned rows; found {len(aligned)}. Check dataset access/config names.')


df = aligned.sample(n=100, random_state=SEED).sort_values('id').reset_index(drop=True)
print('Aligned devtest rows:', len(aligned))
print('Evaluation examples:', len(df))
display(df[['id', 'english', 'indonesian_reference', 'balinese_reference']].head(3))

### 4. Loading the Multilingual Translation Model

BasaLens uses Meta's NLLB-200 Distilled 600M, a pretrained neural machine translation model designed to support a wide range of languages, including Indonesian and Balinese.

The tokenizer is configured with English (`eng_Latn`) as the source language. The model's numerical precision is selected according to the available hardware: FP16 on GPU to reduce memory usage and FP32 on CPU for compatibility.

Finally, the model is moved to the selected device and placed in evaluation mode. This prepares a consistent inference environment for comparing the two target languages.

In [ ]:
MODEL_NAME = 'facebook/nllb-200-distilled-600M'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_dtype = torch.float16 if device.type == 'cuda' else torch.float32

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, src_lang='eng_Latn')
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, torch_dtype=model_dtype)
model.to(device)
model.eval()

print('Model loaded on:', device)

### 5. Translation Inference and Decoding Comparison

This stage implements the core experimental procedure: translating the same 100 English sentences into Indonesian and Balinese using NLLB-200.

A reusable translation function handles both target languages, processes inputs in batches, and applies the appropriate target-language token. Inference mode reduces unnecessary computational overhead during generation.

Two decoding strategies are evaluated:

- **Greedy decoding (`num_beams=1`):** selects the most probable next token at each generation step.
- **Beam search (`num_beams=4`):** explores multiple candidate sequences to identify a higher-scoring output.

This produces 400 translations in total: 100 sentences × 2 target languages × 2 decoding strategies. Keeping the source sentences and model constant allows the experiment to examine how target language and decoding strategy affect translation performance.

In [ ]:
def translate_many(texts, target_language, num_beams=4, batch_size=8):
    if target_language not in {'ind_Latn', 'ban_Latn'}:
        raise ValueError('Unsupported target language code.')
    tokenizer.src_lang = 'eng_Latn'
    target_id = tokenizer.convert_tokens_to_ids(target_language)
    if target_id is None or target_id == tokenizer.unk_token_id:
        raise ValueError(f'Could not resolve target language token: {target_language}')

    predictions = []
    for start in range(0, len(texts), batch_size):
        batch_texts = [str(x) for x in texts[start:start + batch_size]]
        encoded = tokenizer(
            batch_texts, return_tensors='pt', padding=True,
            truncation=True, max_length=512
        )
        encoded = {key: value.to(device) for key, value in encoded.items()}
        with torch.inference_mode():
            generated = model.generate(
                **encoded,
                forced_bos_token_id=target_id,
                num_beams=num_beams,
                max_new_tokens=128
            )
        predictions.extend(tokenizer.batch_decode(generated, skip_special_tokens=True))
    return predictions

# 100 English sentences × 2 target languages × 2 decoding settings = 400 translations.
for target_code, short_name in [('ind_Latn', 'indonesian'), ('ban_Latn', 'balinese')]:
    df[f'{short_name}_greedy'] = translate_many(df['english'].tolist(), target_code, num_beams=1)
    df[f'{short_name}_beam4'] = translate_many(df['english'].tolist(), target_code, num_beams=4)

display(df[['english', 'indonesian_beam4', 'balinese_beam4']].head(5))

### 6. Quantitative Evaluation Using chrF

To measure translation quality consistently, BasaLens applies the chrF evaluation metric from SacreBLEU. chrF compares generated translations with human-written reference translations by measuring character n-gram overlap.

The evaluation is performed at the corpus level across all 100 sentences for four experimental conditions:

1. Indonesian using greedy decoding.
2. Indonesian using beam search with four beams.
3. Balinese using greedy decoding.
4. Balinese using beam search with four beams.

The resulting scores provide a quantitative basis for comparing both target languages and decoding strategies under identical evaluation conditions. Higher chrF scores indicate greater character-level similarity to the reference translations; however, they should not be interpreted as translation accuracy percentages or as a complete measure of meaning and naturalness.

In [ ]:
chrf = CHRF(word_order=0)

comparisons = [
    ('Indonesian', 'indonesian_greedy', 'indonesian_reference', 'Greedy'),
    ('Indonesian', 'indonesian_beam4', 'indonesian_reference', 'Beam search (4)'),
    ('Balinese', 'balinese_greedy', 'balinese_reference', 'Greedy'),
    ('Balinese', 'balinese_beam4', 'balinese_reference', 'Beam search (4)'),
]

score_rows = []
for language, prediction_col, reference_col, method in comparisons:
    score = chrf.corpus_score(
        df[prediction_col].tolist(), [df[reference_col].tolist()]
    ).score
    score_rows.append({'Language': language, 'Method': method, 'chrF': round(score, 2)})

scores = pd.DataFrame(score_rows)
display(scores.pivot(index='Language', columns='Method', values='chrF'))

### 7. Visualizing and Exporting Experimental Results

The evaluation scores are converted into a grouped bar chart to make differences between the two target languages and decoding strategies easier to interpret.

The visualization presents the chrF results from the 100-sentence evaluation sample, providing judges with a direct comparison of the four experimental conditions.

The aggregate scores are also exported to `basalens_aggregate_scores.csv` for transparent reporting and further analysis. Only the evaluation results are exported, rather than a copy of the FLORES+ source sentences.

In [ ]:
# Visual comparison for the notebook/demo.
plot_data = scores.pivot(index='Language', columns='Method', values='chrF')
ax = plot_data.plot(kind='bar', figsize=(8, 4))
ax.set_title('NLLB translation quality on 100 held-out FLORES+ examples')
ax.set_ylabel('chrF score (higher is better)')
ax.set_xlabel('Target language')
ax.tick_params(axis='x', rotation=0)
ax.grid(axis='y', alpha=0.25)

# Save aggregate results, not a public copy of the FLORES+ source text.
scores.to_csv('basalens_aggregate_scores.csv', index=False)
print('Saved aggregate score table.')

### 8. Identifying Potential Translation Weaknesses

Beyond overall corpus-level scores, this analysis examines individual translations to identify examples with the lowest sentence-level chrF scores for each target language.

The five lowest-scoring examples are displayed alongside their English source sentences, NLLB-generated translations, and reference translations. This makes it possible to investigate potential weaknesses that may be hidden by aggregate results.

Each example should be examined for possible differences in meaning, word choice, names, numbers, language register, and grammatical structure. These cases are intended to guide qualitative error analysis, rather than automatically classify every low-scoring translation as incorrect.

In [ ]:
for language, prediction_col, reference_col, score_col in [
    ('Balinese', 'balinese_beam4', 'balinese_reference', 'balinese_beam4_chrf'),
    ('Indonesian', 'indonesian_beam4', 'indonesian_reference', 'indonesian_beam4_chrf')
]:
    df[score_col] = [
        chrf.sentence_score(hyp, [ref]).score
        for hyp, ref in zip(df[prediction_col], df[reference_col])
    ]

    print(f'Five lowest-scoring {language} examples:')
    display(
        df.nsmallest(5, score_col)[
            ['english', prediction_col, reference_col, score_col]
        ].rename(columns={prediction_col: 'NLLB output', reference_col: 'Reference', score_col: 'sentence chrF'})
    )

    print('Discuss whether the problem is meaning, word choice, names/numbers, register, or grammar.')

### 9. Human Review of Balinese Translation Quality

Automated metrics alone cannot fully assess whether a translation preserves meaning or sounds natural to native speakers. To complement the chrF evaluation, this stage prepares a sample of up to 25 Balinese translations for qualitative human review.

The sample is selected using a fixed random seed (`17`) to maintain consistency. A simple three-level rubric is provided to assess each translation:

- **0 — Incorrect or unusable:** The translation fails to communicate the intended meaning.
- **1 — Partially correct or awkward:** The general meaning is understandable, but significant issues remain.
- **2 — Clear and natural:** The translation communicates the intended meaning clearly and naturally.

This review provides an additional perspective on translation quality that character-level similarity scores cannot capture. It is exploratory feedback, not a formal or representative evaluation of all Balinese speakers.

In [ ]:
review_sample = df.sample(n=min(25, len(df)), random_state=17).sort_values('id')
display(review_sample[['id', 'english', 'balinese_beam4']].reset_index(drop=True))
print('Reviewer rubric: 0 = wrong/unusable; 1 = partly correct/awkward; 2 = clear and natural.')

### 10. Real-World Application: School Notice Translation

To explore how BasaLens performs beyond benchmark data, this stage tests the model using 10 original English school notices covering everyday school communication.

The notices are translated into Indonesian and Balinese using beam search with four beams. The resulting translations are displayed side by side to examine how the model handles practical expressions, institutional terminology, instructions, and context-dependent vocabulary.

This experiment demonstrates a potential application of multilingual AI translation in educational environments. These examples are exploratory demonstrations and should be reviewed by fluent speakers before being used in official school communication.

In [ ]:
local_notices = [
    'The school library will be closed on Friday.',
    'Please return borrowed books before the end of the week.',
    'The student meeting will begin at eight in the morning.',
    'Parents are invited to attend the school performance.',
    'Please keep the classroom clean after the activity.',
    'The event has been moved to the school hall because of rain.',
    'Students should arrive fifteen minutes before the ceremony.',
    'Please contact the school office if you have any questions.',
    'The lost-and-found desk is beside the main entrance.',
    'Thank you for helping keep our school community safe.'
]
local_df = pd.DataFrame({'english': local_notices})
local_df['nllb_indonesian'] = translate_many(local_notices, 'ind_Latn', num_beams=4, batch_size=4)
local_df['nllb_balinese'] = translate_many(local_notices, 'ban_Latn', num_beams=4, batch_size=4)
display(local_df)

### 11. Interactive Demonstration with Gradio

This stage transforms the trained inference pipeline into an interactive demonstration, allowing judges to test BasaLens using their own English input.

The interface provides two translation targets: Balinese (*Basa Bali*) and Indonesian (*Bahasa Indonesia*). Users can enter a sentence, select the desired language, and receive an AI-generated translation using the same NLLB-200 model and beam-search configuration used in the evaluation.

The Gradio interface makes the research accessible beyond the notebook by allowing direct interaction with the system. It also clearly identifies the output as an AI translation draft, emphasizing the importance of fluent-speaker review for important communication.

In [ ]:
import gradio as gr

def demo_translate(text, target_language):
    if not text or not text.strip():
        return 'Please enter a sentence first.'
    target_code = {
        'Balinese (Basa Bali)': 'ban_Latn',
        'Indonesian (Bahasa Indonesia)': 'ind_Latn'
    }[target_language]
    return translate_many([text.strip()], target_code, num_beams=4, batch_size=1)[0]

demo = gr.Interface(
    fn=demo_translate,
    inputs=[
        gr.Textbox(lines=4, label='English source text', placeholder='Type a short school or community notice...'),
        gr.Dropdown(
            choices=['Balinese (Basa Bali)', 'Indonesian (Bahasa Indonesia)'],
            value='Balinese (Basa Bali)', label='Translate into'
        )
    ],
    outputs=gr.Textbox(lines=5, label='AI translation draft'),
    title='BasaLens',
    description='A research prototype comparing AI translation into Balinese and Indonesian. Have a fluent speaker review important text.'
)
demo.launch(share=True)